In [ ]:
import os
import cv2
import re
import json
import pandas as pd
import pytesseract
import textdistance
from google.cloud import vision
import requests
from rapidfuzz import fuzz

# ================= CONFIG =================
GCP_CRED_PATH = "vision_api.json"   # <-- put this file in same folder
MASTER_CSV = "Medicine_Details_revised_new.csv"

if not os.path.exists(GCP_CRED_PATH):
    raise FileNotFoundError("Google Vision credentials missing.")

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GCP_CRED_PATH
vision_client = vision.ImageAnnotatorClient()

LLM_MODEL = "mistralai/mistral-7b-instruct"
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")  # SAFER

if not OPENROUTER_API_KEY:
    raise ValueError("Set OPENROUTER_API_KEY as environment variable")

# ================= ROUTE MAP =================
route_map = {
    "cap": "oral", "tablet": "oral", "capsule": "oral",
    "syrup": "oral", "injection": "injectable",
    "ointment": "topical", "cream": "topical",
    "gel": "topical", "drops": "ocular"
}

term_normalizer = {
    "tas": "tab", "taz": "tab", "tab.": "tab",
    "drip": "drops", "dryps": "drops", "drips": "drops"
}

# ================= IMAGE PROCESSING =================
def extract_bracket_regions(image_path, output_dir="bracket_groups"):
    os.makedirs(output_dir, exist_ok=True)
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    img_h, img_w = gray.shape

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 30))
    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    crops = []
    for i, cnt in enumerate(contours):
        x, y, w, h = cv2.boundingRect(cnt)
        if h > 60:
            crop = image[y:y+h, :]
            fname = f"{output_dir}/block_{i}.png"
            cv2.imwrite(fname, crop)
            crops.append(fname)

    return crops

# ================= OCR =================
def get_ocr_output(folder):
    output = []
    for fname in os.listdir(folder):
        with open(os.path.join(folder, fname), "rb") as f:
            img = vision.Image(content=f.read())
        res = vision_client.text_detection(image=img)
        txt = res.text_annotations[0].description if res.text_annotations else ""
        output.append((fname, txt))
    return output

# ================= LLM =================
def extract_meds_from_llm(text):
    prompt = f"""
Extract medicines as JSON:
[
  {{
    "medicine": "...",
    "route": "...",
    "dosage and duration": "..."
  }}
]
TEXT:
{text}
"""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": LLM_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2
    }

    r = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers=headers, json=payload
    )
    r.raise_for_status()
    content = r.json()["choices"][0]["message"]["content"]

    start, end = content.find("["), content.rfind("]") + 1
    return json.loads(content[start:end])

# ================= MAIN PIPELINE =================
def run_pipeline(image_path):
    extract_bracket_regions(image_path)
    ocr = get_ocr_output("bracket_groups")

    master_df = pd.read_csv(MASTER_CSV)
    master_meds = master_df["Medicine Name"].astype(str).tolist()

    results = []
    for _, text in ocr:
        preds = extract_meds_from_llm(text)
        results.extend(preds)

    return results

# ================= BATCH RUNNER =================
def run_on_dataset(images_dir):
    os.makedirs("predictions", exist_ok=True)

    for img in sorted(os.listdir(images_dir)):
        if not img.lower().endswith((".jpg", ".png", ".jpeg")):
            continue

        img_id = os.path.splitext(img)[0]
        print(f"Running on {img}")

        result = run_pipeline(os.path.join(images_dir, img))

        with open(f"predictions/{img_id}.json", "w") as f:
            json.dump(result, f, indent=2)

        print(f"Saved predictions/{img_id}.json")

# ================= ENTRY POINT =================
if __name__ == "__main__":
    run_on_dataset("dataset_hp_acc1/images")


 Preprocessing image and extracting text regions...
 Running OCR...
 Loading master medicine list...

--- Processing bracket_1.png ---
 Failed to parse JSON from LLM response:
  <s> [OUT] [
  {
    "medicine": "Pantoprazole",
    "route": "oral",
    "dosage and duration": "1 tablet in the morning for 1 month"
  },
  {
    "medicine": "Pantoprazole",
    "route": "oral",
    "dosage and duration": "1 tablet at night for 1 month"
  }
] 

--- Processing bracket_2.png ---

--- Processing bracket_3.png ---
 Failed to parse JSON from LLM response:
  <s> 

--- Processing bracket_4.png ---
 Failed to parse JSON from LLM response:
  <s> [OUT] [
  {
    "medicine": "paracetamol",
    "route": "oral",
    "dosage and duration": "1 tablet in the morning and night for 5 days"
  },
  {
    "medicine": "amoxicillin",
    "route": "oral",
    "dosage and duration": "1 tablet in the morning and night for 5 days"
  },
  {
    "medicine": "omeprazole",
    "route": "oral",
    "dosage and duration": "1 